# 05. Multi-Turn Chat & Python Function Calling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/en/05_chat_and_function_calling.ipynb)

The `google-genai` SDK turns stateless model requests into stateful, tool-augmented assistants using two built-in primitives: **`client.chats.create()`** for managing multi-turn conversation history, and **Automatic Python Function Calling (`tools=[fn]`)**, where the SDK introspects Python type hints and docstrings, executes local functions when Gemini requests them, and feeds the structured results back automatically.

![Figure 5-1: Automatic Python Function Calling Execution Loop in google-genai](https://examples.drawlib.com/google-genai/images/05_chat_and_function_calling/ch05_function_calling_loop.png)

*Figure 5-1: Automatic Python Function Calling Execution Loop in google-genai*

---

## 1. 3-Line Client Setup & Stateful Multi-Turn Chat (`client.chats.create`)

Start by reading your `GEMINI_API_KEY` from Google Colab Secrets (`userdata.get`) and creating a unified `genai.Client` instance:

In [ ]:
!pip install -q drawlib google-genai pydantic

In [ ]:
from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

While `client.models.generate_content()` is stateless, **`client.chats.create()`** maintains an in-memory `list[types.Content]` transcript across turns. Each call to `chat.send_message()` appends a `Content(role="user")` message, sends the accumulated history to Gemini, and records the returned `Content(role="model")` turn (along with any intermediate tool calls). Because long multi-turn sessions resend a growing shared prefix on every turn, they automatically benefit from Gemini 2.5's **Implicit Context Caching**, lowering time-to-first-token (TTFT) and input token cost on repeated prefixes with zero extra configuration.

![Figure 5-2: Multi-Turn Chat History Accumulation and Tool-Augmented Session Flow](https://examples.drawlib.com/google-genai/images/05_chat_and_function_calling/ch05_chat_state_context.png)

*Figure 5-2: Multi-Turn Chat History Accumulation and Tool-Augmented Session Flow*

Create a stateful chat session with `gemini-2.5-flash`, send two sequential follow-up prompts, and inspect `chat.get_history()` to see how `user` and `model` turns accumulate:

In [ ]:
chat = client.chats.create(model="gemini-2.5-flash")

reply_1 = chat.send_message(
    "We are deploying a multi-region Cloud Run API gateway. Give 2 short architecture tips."
)
print("Model (Turn 1):\n", reply_1.text)

reply_2 = chat.send_message(
    "Based on those two tips, what HTTP timeout and retry budget should our client use?"
)
print("\nModel (Turn 2):\n", reply_2.text)

print("\n--- Accumulated Session History ---")
for idx, turn in enumerate(chat.get_history(), start=1):
    preview = " ".join(
        p.text.strip() for p in (turn.parts or []) if p.text and not p.thought
    )
    print(f"[{idx}] role={turn.role:<5} | {preview[:85]}...")

---

## 2. Automatic Python Function Calling & Inline Drawlib Visualization

To give Gemini live access to external metrics, databases, or internal APIs, pass plain typed Python functions directly into **`types.GenerateContentConfig(tools=[...])`**. Be sure to include **type annotations** (`region: str -> dict[str, float]`) and a clear **Google-style docstring** (`Args:` and `Returns:`): the SDK compiles them into a JSON Schema tool declaration automatically.

When Gemini decides it needs telemetry for multiple regions, `genai.Client` executes `get_region_latency()` locally for each requested region, packages the returned Python dictionaries into `FunctionResponse` payloads, and returns the final synthesized recommendation:

In [ ]:
from google.genai import types

REGION_LATENCY_MS: dict[str, dict[str, float]] = {
    "asia-northeast1": {"p50_ms": 14.2, "p95_ms": 31.5},
    "us-central1": {"p50_ms": 24.5, "p95_ms": 58.4},
    "europe-west1": {"p50_ms": 38.6, "p95_ms": 84.2},
}


def get_region_latency(region: str) -> dict[str, float]:
    """Fetch live p50 and p95 API latency in milliseconds for a cloud region.

    Args:
        region: Cloud region ID ('asia-northeast1', 'us-central1', or 'europe-west1').

    Returns:
        Dictionary with 'p50_ms' median and 'p95_ms' tail latency values.
    """
    return REGION_LATENCY_MS.get(region, {"p50_ms": -1.0, "p95_ms": -1.0})


tool_chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        tools=[get_region_latency],
        temperature=0.1,
    ),
)

recommendation = tool_chat.send_message(
    "Check the p50 and p95 latency across asia-northeast1, us-central1, and europe-west1, "
    "then recommend the best primary and failover regions for users in Tokyo."
)
print(recommendation.text)

print("\n--- Recorded Tool Call Trace ---")
for turn in tool_chat.get_history():
    for part in turn.parts or []:
        if part.function_call:
            print(f"-> FunctionCall: {part.function_call.name}({dict(part.function_call.args or {})})")
        elif part.function_response:
            print(f"<- FunctionResponse: {part.function_response.response}")

Finally, render the exact multi-region `p50` and `p95` latency metrics queried by the agent directly inside your Jupyter or Colab output cell using Drawlib's **`BarChart`** with `setup(..., cwd=True)` and `show()`:

In [ ]:
from drawlib.canvas import clear, setup, show
from drawlib.charts.bar import BarChart
from drawlib.styles import Styles
from drawlib.text import text

clear()
setup(width=116, height=56, cwd=True)

regions = list(REGION_LATENCY_MS.keys())
p50_ms = [get_region_latency(r)["p50_ms"] for r in regions]
p95_ms = [get_region_latency(r)["p95_ms"] for r in regions]

text(
    (58.0, 50.8),
    "Multi-Region API Latency Queried via Gemini Function Calling",
    style=Styles.DarkBold.patch(text_size=11.5),
)

chart = BarChart(
    axis_line_style=Styles.MutedDashed,
    categories=regions,
    width=96.0,
    height=34.0,
    orientation="vertical",
    bar_mode="group",
    bar_width_ratio=0.62,
    bar_r=0.8,
    axis_text_style=Styles.Dark.patch(text_size=10.0),
    grid_style=Styles.MutedThin,
    value_text_style=Styles.DarkBold.patch(text_size=10.0),
    value_format="{:.1f} ms",
)
chart.add_series("p50 Median (ms)", p50_ms, style=Styles.PrimaryFlat)
chart.add_series("p95 Tail (ms)", p95_ms, style=Styles.SecondaryNeutral)
chart.configure_y_axis(min_value=0.0, max_value=100.0, tick_step=25.0, unit=" ms")
chart.draw(xy=(12.0, 6.0))
chart.draw_legend(
    xy=(36.0, 44.5),
    text_style=Styles.Dark.patch(text_size=10.0),
    orientation="horizontal",
)

show()

> **Key Takeaways**: Combine `client.chats.create()` for stateful multi-turn context, typed Python functions in `tools=[fn]` for zero-boilerplate tool execution, and `setup(..., cwd=True)` + `show()` to turn live Gemini tool telemetry into publication-grade Drawlib charts right inside your notebook.